In [2]:
import copy
import tiktoken

#from settings import Settings
#from models import Question
from src.services.OpenAIClient import (
    OpenAIClient,
    Gpt4o,
    Gpt4oMini,
    #Gpto1Mini,
    #Gpto1Preview,
    Gpto1,
    Gpto3,
)

from src.services.accuracy import calculate_accuracy
from src.services.read_excel import read_excel
##from src.services.recorder import ApiHistoryRecorder
from src.strategies.solve_questions_by_openai import (
    BasicSolveQuestionPrompt,
    solve_questions_by_openai_independently,
    solve_type_d_questions_by_openai_dependently,
    OptimizedSolveQuestionPrompt1,
    NormalSolveQuestionPrompt,
    OptimizedSolveQuestionPrompt2_for_essential_a_b,
    OptimizedSolveQuestionPrompt2_for_c_d
)
from src.strategies.translate_to_English_by_openai import (
    BasicTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt1,
    NormalTranslateToEnglishPrompt,
    OptimizedTranslateToEnglishPrompt,
)

In [193]:
record_excel_path = "../record/record.xlsx"
file_path = "../problems/76th/essential"
#file_path = "../problems/76th/academic_a"

problem_file_path = f'{file_path}/questions.xlsx'
answer_file_path = f'{file_path}/answers.xlsx'

### =====変更=====
model = Gpto3

is_solving_prompt_normal = True

EXECUTE_NUM = 3

is_translated_to_English = False

solve_question_prompt = NormalSolveQuestionPrompt if is_solving_prompt_normal else OptimizedSolveQuestionPrompt2_for_c_d # OptimizedSolveQuestionPrompt1
translate_to_english_prompt = OptimizedTranslateToEnglishPrompt
is_image_contained = False
### ============

WAIT_SEC = 15
solve_num = -1
batch_size = 10


is_dry_run = False

is_d = file_path[-1] == 'd'


assert is_image_contained == (file_path[-1] in {'c', 'd'})
assert EXECUTE_NUM <= 3



In [194]:
#api_history_recorder = ApiHistoryRecorder(excel_path=record_excel_path)
openai_client = OpenAIClient(
    model=model,
    #api_history_recorder=api_history_recorder,
)

In [195]:
questions = read_excel(file_path=file_path)

for i in range(4):
    print(f'======question:{i+1}======')
    print(questions[i].type_d_common_sentence)
    print(questions[i].question_sentence)
    print(questions[i].answer_options)
    print(questions[i].correct_answer)
    print(questions[i].image_path)


======question:1======
None
獣医師の行動として適切でないのはどれか。
１．状態の悪い動物が入院していたため、雇用している獣医師１名を指名し、１ 週間付き切りで勤務させた。 ２．薬剤耐性菌を有する外傷症例であったため、入院は他の動物と一緒の部屋に してほしいと言う飼い主の申し出を断った。 ３．去勢手術の予定があったが、午前中の診察が長引いたので、飼い主の同意を 得て手術を翌日に延期した。 ４．自院の設備が不十分で、治療リスクが高い症例を他の適切な獣医療施設に紹 介した。 ５．保険請求のため、今回治療した先天性疾患を後天的なものとして診断書を書 いてほしいという飼い主の要求を断った。
{<AnswerEnum.CHOICE_1: 1>}
None
======question:2======
None
実験動物の福祉と倫理に関する記述として適切でないのはどれか。
１．本来の習性に近い行動ができる環境での飼養が望ましい。 ２．疾患モデル動物では発症によって相応の苦痛が生じると考えるべきである。 ３．実験に支障を及ぼさない範囲で麻酔や鎮痛処置を行う。 ４．実験に使う動物数はできる限り少なくすべきである。 ５．麻酔薬を用いて安楽殺する場合は動物の状態にかかわらず時間をかけて投与 する。
{<AnswerEnum.CHOICE_5: 5>}
None
======question:3======
None
養鶏場から飼養家きんの死亡が急激に増えているとの連絡があった。同事実を確認した獣医師が最初にとるべき行動として最も適当なのはどれか。
１．同様の症状がないか、近隣の養鶏場に直行して聞き取りをする。
２．死亡鶏を死亡獣畜取扱場に運搬するよう指示する。
３．異常がみられない鶏を近隣の養鶏場に避難させるよう指示する。
４．管轄の家畜保健衛生所に連絡する。
５．死亡数が減少するまで様子を見るよう指示する。
{<AnswerEnum.CHOICE_4: 4>}
None
======question:4======
None
獣医療に関する記述として正しいのはどれか。
１．急患では診療簿に診療を行った年月日のみを記載すればよい。 ２．友人の犬を私的に避妊手術した場合は診療簿の作成は不要である。 ３．診療簿には診療した動物の所有者の氏名と住所を記載する必要があ

In [196]:
import asyncio

for num in range(EXECUTE_NUM):
    question_temp = copy.deepcopy(questions[:solve_num] if solve_num >= 0 else questions)
    if is_d:
        questions_res = await solve_type_d_questions_by_openai_dependently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    else:
        questions_res = await solve_questions_by_openai_independently(
            openai_client=openai_client,
            questions=question_temp,
            is_translated_to_English=is_translated_to_English,
            excel_output_path=problem_file_path,
            solve_question_prompt=solve_question_prompt,
            translate_to_english_prompt=translate_to_english_prompt,
            is_image_contained=is_image_contained,
            batch_size=batch_size,
            is_dry_run=is_dry_run,
            does_also_write_openai_answer=True,
        )
    if num < EXECUTE_NUM-1:
        await asyncio.sleep(WAIT_SEC)


[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '獣医師の行動として適切でないのはどれか。 The answer options are １．状態の悪い動物が入院していたため、雇用している獣医師１名を指名し、１ 週間付き切りで勤務させた。 ２．薬剤耐性菌を有する外傷症例であったため、入院は他の動物と一緒の部屋に してほしいと言う飼い主の申し出を断った。 ３．去勢手術の予定があったが、午前中の診察が長引いたので、飼い主の同意を 得て手術を翌日に延期した。 ４．自院の設備が不十分で、治療リスクが高い症例を他の適切な獣医療施設に紹 介した。 ５．保険請求のため、今回治療した先天性疾患を後天的なものとして診断書を書 いてほしいという飼い主の要求を断った。. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '実験動物の福祉と倫理に関する記述として適切でないのはどれか。 The answer options are １．本来の習性に近い行動ができる環境での飼養が望ましい。 ２．疾患モデル動物では発症によって相応の苦痛が生じると考えるべきである。 ３．実験に支障を及ぼさない範囲で麻酔や鎮痛処置を行う。 ４．実験に使う動物数はできる限り少なくすべきである。 ５．麻酔薬を用いて安楽殺する場合は動物の状態にかかわらず時間をかけて投与 する。. Respond with only the number of your choice (e.g., 1, 2, 3, etc.).'}]
[{<Roles.system: 'system'>: 'Please answer this question.'}, {<Roles.user: 'user'>: '養鶏場から飼養家きんの死亡が急激に増えているとの連絡があった。同事実を確認した獣医師が最初にとるべき行動として最も適当なのはどれか。 The answer option

In [197]:
Qs = questions_res if solve_num < 0 else questions_res[:solve_num]
token_sum = 0

tiktoken_model = "gpt2"

def cal_token_num(sentence):
    enc = tiktoken.get_encoding(tiktoken_model)
    tokens = enc.encode(sentence)
    return len(tokens)

for q in Qs:
    if is_translated_to_English:#not q.is_correct():
        print(q.question_sentence)
        print(q.answer_options)
        print(q.question_sentence_in_English)
        print(q.openai_answer)
        print(q.correct_answer)
        #question_token_num = cal_token_num(q.question_sentence_in_English)
        #options_token_num = cal_token_num(q.answer_options_in_English)
        #token_sum += question_token_num + options_token_num

print(token_sum)


0


In [198]:
import datetime

dt_now = datetime.datetime.now()
print(dt_now.strftime('%Y/%m/%d %H:%M'))
print(f"model:{openai_client.model.model}")
score = calculate_accuracy(questions=questions_res)
print(f"score:{score}")

2025/05/17 14:02
model:o3
score:{'accuracy': 0.96, 'correct_num': 48, 'wrong_num': 2}


In [199]:
"""
from src.services.image_encoder import pdf_encoder_in_base64
base64_image = pdf_encoder_in_base64("../problems/74th/academic_c/images/image1.PDF")

openai_client2 = OpenAIClient(model=Gpt4oMini)

answer =  await openai_client2.fetch_completion(
    system_prompt='you are vet.',
    user_prompt="""
#    これはなんの写真ですか？
""",
    base64_image=base64_image,
)

print(answer)
"""

',\n    base64_image=base64_image,\n)\n\nprint(answer)\n'

In [8]:
system_prompt = OptimizedSolveQuestionPrompt2_for_essential_a_b.system_prompt
WIDTH = 70
cnt = 0
for ch in list(system_prompt.split()):
    print(ch, end=' ')
    cnt += len(ch)
    if cnt >= WIDTH:
        cnt = 0
        print()

You are tasked with answering this veterinary question. This is an examination in Japan. 
Therefore, you must refer to the laws, guidelines, and political system in Japan. Think 
deeply and thoroughly. Choose the best possible answer from the given options. Do not 
include explanations or additional text in the output. 